# 11.18 指定哪一塊文字，如何只讀那一塊？


這張字卡左邊寫「入口」，右邊寫「出口」。如果問題是「只讀右側框內的字」，希望答案是「出口」，不需要把左邊一起抄出來。先看清兩區的內容，再看綠色框指定了哪一區。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/new-11.18-two-regions.svg")))

字卡是作者製作的可見材料。「入口」和「出口」都只用到 MoE 助理有限字表裡的字，這一題也不用先會讀任意街景招牌。框線提供位置，不提供字詞答案：把右框裡的內容換成「開關」，同一條「只讀右側框內的字」就應得到「開關」。

這個工作可以拆成兩個理解步驟：先選出使用者指定的區域，再辨認該區的字。選區回答「我要看哪裡」，讀字回答「那裡寫了什麼」。假如選到左區，卻完整讀出「入口」，辨字可能沒錯，指定範圍仍然錯了。假如選到右區，卻寫成「出入」，選區正確，字詞內容或次序仍然錯了。

讓模型只讀指定的一塊，有兩種安排。第一種是先裁切再讀：使用者給框，固定的裁切步驟按框的座標取出框內像素，只把這塊像素送給自行訓練的讀字入口（專門把字卡像素轉成特徵的圖片入口），再交給文字核心生成字串。裁切只按座標拿出一塊圖，不會認字。第二種是整張圖加框：模型看整張字卡和框的位置，自己學會只使用框內的區域。本節採用第一種：它把「指定哪一區」交給已知的框，讓我們先專心教少量字形和字序。不要把固定位置的裁切說成「會自動找到任何圖片裡的文字」；自動尋找文字位置是另一個任務，需要相應的材料與檢查。

下面用 [11.12](https://birdhackor.github.io/tiny-perceptron-vlm/11.12.html) 的兩個數字字形代替「入口」「出口」：中文字形要靠字型檔才能畫成像素，而框與裁切的作用和字的種類無關。程式把兩個字形左右排成一張小卡，中間隔一欄空白，再按框的欄位裁切。


In [ ]:
import torch

glyph = {
    "0": torch.tensor([[1, 1, 1], [1, 0, 1], [1, 0, 1], [1, 0, 1], [1, 1, 1]]),
    "1": torch.tensor([[0, 1, 0], [1, 1, 0], [0, 1, 0], [0, 1, 0], [1, 1, 1]]),
}
gap = torch.zeros(5, 1, dtype=torch.long)
card = torch.cat([glyph["0"], gap, glyph["1"]], dim=1)  # 左區寫 0、右區寫 1
swapped = torch.cat([glyph["1"], gap, glyph["0"]], dim=1)  # 兩區內容互換
box = {"左框": (0, 3), "右框": (4, 7)}  # 每個框記起訖欄：start 到 end-1
print("字卡形狀", tuple(card.shape))
for title, image, name in [("原卡", card, "右框"), ("原卡", card, "左框"), ("互換後", swapped, "右框")]:
    start, end = box[name]
    crop = image[:, start:end]  # 固定裁切：只按座標取像素，不認字
    matched = [label for label, pixels in glyph.items() if torch.equal(crop, pixels)]
    print(title, name, "框內是字形", matched)

`torch.cat([...], dim=1)` 沿欄的方向，把左字形、一欄空白、右字形接在一起，得到 5 列×7 欄的字卡。`box` 用起訖欄記兩個框：右框是第 4 到 6 欄，切片 `start:end` 不含 end（同 [11.9](https://birdhackor.github.io/tiny-perceptron-vlm/11.9.html)）。迴圈依序做三個對照，`image[:, start:end]` 取出框內像素，這就是固定的裁切步驟，它只看座標。下一行逐一取出字典裡的字形，只留下和 `crop` 完全相同（`torch.equal`）的標籤。這個核對只是讓我們看見「送給讀字入口的是哪些像素」，不是讀字入口；真正的讀字入口要從這塊像素學出字串。

「字卡形狀」印 `(5,7)`；之後的三行輸出就是三個對照：原卡的右框是字形 1，左框是字形 0；兩區內容互換後，同一個右框變成字形 0。框只決定取哪裡，內容換了，送出去的像素也跟著換，所以正確答案必須跟著改。

採用先裁切再讀時，每筆訓練示範就是「裁切後的框內像素＋正確字串」，例如右框裁出的「出口」像素配答案「出口」。選整張圖加框時，示範才要把整張字卡、框的位置和正確字串放在同一筆裡。可以先用已知字表的單字與短詞，讓模型學字形對應，再增加少量字的組合。若總把左右文字一起寫進答案，模型學到的就會是整張抄寫；答案範圍必須和要求一致。

真正的字卡也用同樣三個對照檢查：指定右框，應得到「出口」；改指定左框，應得到「入口」；保持右框不動、把兩區內容互換，應得到「入口」。改指定左框看答案會不會跟著框改變，互換內容看答案會不會跟著框內的字改變；答錯時要把這兩個結果合起來看，才分得出是哪一種捷徑。以整張圖加框為例：一律答「出口」的模型兩個都錯；總是讀右區、不理會框的模型只錯在改指定左框；按框的位置記答案（右框答「出口」、左框答「入口」）、不看字形的模型只錯在互換內容。三個都答對，就排除了這三種捷徑。

這些對照也讓錯誤更容易解讀。「入口出口」包含正確字詞，卻多抄了框外內容；「出口」符合內容與範圍；「口出」用了相同兩個字，次序仍不符合原圖。按 [11.16](https://birdhackor.github.io/tiny-perceptron-vlm/11.16.html) 的算法，「入口出口」要刪兩個字、「口出」要換兩個字，CER 都是 2/2＝1。前面已學過逐字與字序的檢查，這裡多加的是選取範圍，而不是重新發明一種讀字分數。

驗收有限字卡時，還要準備沒練過的字體或版面變體當測試題，並試已知字元組成的新短串。這樣才知道模型是否只記住一張「出口」圖。讀過這張清晰字卡，也不足以推出它能閱讀模糊、傾斜或沒見過的字組成的真實招牌。

練習把程式中右框的 `(4, 7)` 改成 `(3, 6)`，也就是往左偏一欄，先預測兩個右框的輸出：框裡變成一欄空白加右字形的前兩欄，不等於任何一張完整字形，所以印空清單 `[]`；左框不受影響，仍是 `['0']`。框畫偏了，即使讀字入口很強，送進去的也是不完整的字。

<details>
<summary>字型來源與前文連結</summary>

字卡短詞與希望答案由作者選寫，是教材設計例；SVG 指定 Noto Sans CJK TC 並提供系統中文字型作備援。Noto CJK 的[官方字型授權](https://github.com/notofonts/noto-cjk/blob/main/Sans/LICENSE)是 SIL OFL 1.1。使用字型產生文件與重新散佈字型檔，是不同的授權情況；本節沒有打包字型檔。

[11.15](https://birdhackor.github.io/tiny-perceptron-vlm/11.15.html)說明筆畫資訊，[11.16](https://birdhackor.github.io/tiny-perceptron-vlm/11.16.html)說明逐字與順序；本節再問指定框是否改變應讀的內容。

</details>
